# ⚡ SoulX-FlashHead — Inference on RunPod GPUs

This notebook walks through the full pipeline for **[SoulX-FlashHead](https://github.com/Soul-AILab/SoulX-FlashHead)**,
an audio-driven real-time streaming talking-head model, end-to-end on a RunPod GPU pod:

1. Check the GPU / environment
2. Clone the repository
3. Install dependencies (PyTorch, requirements, FlashAttention, FFmpeg)
4. Download the model weights from Hugging Face
5. Run inference — **Lite** (single GPU), **Pro** (single GPU), and **Pro** (multi-GPU)
6. Preview the generated video
7. (Optional) Launch the Gradio demos

---

### 🎯 GPU guidance (from the model card)

| Model | Single GPU | Real-time (25+ FPS) |
| :-- | :-- | :-- |
| **Lite** | ~96 FPS on RTX 4090 | ✅ single RTX 4090 (up to 3 concurrent) |
| **Pro** | ~10.8 FPS on RTX 4090 | ✅ two RTX 5090 (with SageAttention) |

If you only have one GPU, start with the **Lite** model. The **Pro multi-GPU** cell needs a pod with **2+ GPUs**.

> **RunPod tip:** Work inside `/workspace` — it is the persistent volume, so your repo, models, and outputs survive pod restarts. Pick a template with **CUDA 12.8** (the build targets `cu128`).

## 1. Check the GPU & environment

In [1]:
# What GPU(s) do we have, and how much VRAM?
!nvidia-smi

Mon Jun  8 03:46:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          On  |   00000000:AF:00.0 Off |                    0 |
| N/A   26C    P0             26W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
import sys, platform
print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())

try:
    import torch
    print("torch   :", torch.__version__)
    print("CUDA    :", torch.version.cuda)
    print("GPUs    :", torch.cuda.device_count())
except ImportError:
    print("torch   : not installed yet (will be installed below)")

Python  : 3.12.3
Platform: Linux-6.8.0-107-generic-x86_64-with-glibc2.39
torch   : 2.8.0+cu128
CUDA    : 12.8
GPUs    : 1


## 2. Clone the repository

We clone into `/workspace` so everything lives on the persistent volume. All later cells assume the
working directory is the repo root.

In [3]:
import os

WORKSPACE = "/workspace" if os.path.isdir("/workspace") else os.path.expanduser("~")
REPO_DIR  = os.path.join(WORKSPACE, "SoulX-FlashHead")
os.chdir(WORKSPACE)

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Soul-AILab/SoulX-FlashHead.git
else:
    print("Repo already present, pulling latest...")
    !cd "{REPO_DIR}" && git pull

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())
!ls -la

Cloning into 'SoulX-FlashHead'...
remote: Enumerating objects: 163, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (55/55), done.
remote: Total 163 (delta 37), reused 25 (delta 18), pack-reused 90 (from 1)
Receiving objects: 100% (163/163), 6.77 MiB | 13.84 MiB/s, done.
Resolving deltas: 100% (48/48), done.
Working directory: /workspace/SoulX-FlashHead
total 100
drwxr-xr-x 6 root root  4096 Jun  8 03:46 .
drwxr-xr-x 5 root root   135 Jun  8 03:46 ..
drwxr-xr-x 8 root root  4096 Jun  8 03:46 .git
-rw-r--r-- 1 root root   126 Jun  8 03:46 .gitignore
-rw-r--r-- 1 root root 11357 Jun  8 03:46 LICENSE
-rw-r--r-- 1 root root 11284 Jun  8 03:46 README.md
drwxr-xr-x 2 root root   156 Jun  8 03:46 assets
drwxr-xr-x 2 root root    65 Jun  8 03:46 examples
drwxr-xr-x 8 root root   168 Jun  8 03:46 flash_head
-rw-r--r-- 1 root root  8666 Jun  8 03:46 generate_video.py
-rw-r--r-- 1 root root 17028 Jun  8 03:46 gradio_app.py
-rw-r--r-- 1 root root 12761 Jun  8 

## 3. Install dependencies

The order matters: **PyTorch first**, then the project requirements, then FlashAttention (which builds
against the installed torch).

### 3a. PyTorch (CUDA 12.8 build)

The repo is pinned to `torch==2.7.1` + `cu128`, which is what `xformers==0.0.31` and `flash_attn==2.8.0.post2`
in `requirements.txt` expect. If your RunPod base image already ships this exact combo you can skip this cell,
but installing it explicitly avoids subtle ABI mismatches.

In [4]:
!pip install torch==2.7.1 torchvision==0.22.1 --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 64.2 MB/s  0:00:01 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 156.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 121.6 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 726.9/726.9 MB 91.3 MB/s  0:00:07:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 609.6/609.6 MB 106.6 MB/s  0:00:06a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 105.0 MB/s  0:00:0100:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 108.1 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.4/260.4 MB 107.3 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 292.1/292.1 MB 103.2 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 156.8/156.8 MB 105.0 MB/s  0:00:0100:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.3/201.3 

### 3b. Project requirements

This installs everything in the repo's `requirements.txt` (diffusers, transformers, gradio, xfuser, xformers,
librosa, mediapipe, loguru, etc.).

In [5]:
!pip install -r requirements.txt

ERROR: Could not find a version that satisfies the requirement mediapipe==0.10.9 (from versions: 0.10.13, 0.10.14, 0.10.15, 0.10.18, 0.10.20, 0.10.21, 0.10.30, 0.10.31, 0.10.32, 0.10.33, 0.10.35)
ERROR: No matching distribution found for mediapipe==0.10.9


### 3c. FlashAttention

Building FlashAttention from source can take **20–40 minutes**. `MAX_JOBS` is capped to avoid running the
host out of RAM during compilation.

> **Faster alternative:** download a prebuilt wheel matching your stack (`cp310`, `torch2.7`, `cu12`) from the
> [FlashAttention releases page](https://github.com/Dao-AILab/flash-attention/releases/tag/v2.8.0.post2) and
> `pip install` it directly — pick the file whose `cxx11abi` flag matches your torch build. Example:
> ```bash
> pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.0.post2/flash_attn-2.8.0.post2+cu12torch2.7cxx11abiFALSE-cp310-cp310-linux_x86_64.whl
> ```

In [6]:
import os
os.environ["MAX_JOBS"] = "4"  # limit parallel compile jobs to avoid OOM

!pip install ninja
!pip install flash_attn==2.8.0.post2 --no-build-isolation

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.9/7.9 MB 8.9 MB/s  0:00:00 eta 0:00:01
  Preparing metadata (pyproject.toml) ... done
  Created wheel for flash_attn: filename=flash_attn-2.8.0.post2-cp312-cp312-linux_x86_64.whl size=255941970 sha256=fc596b041e6d43cd00886a4715234accd75a8e93813c8ed0ad17b47466693610
  Stored in directory: /workspace/.cache/pip/wheels/0f/fe/8a/ba201a3988f670e6cadd9290c790ff3278198095d368883ddd
Successfully built flash_attn
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [flash_attn]2 [flash_attn]


### 3d. SageAttention (optional)

Only needed for the fastest **Pro** real-time path (e.g. two RTX 5090). Safe to skip otherwise.

In [7]:
# Optional — uncomment to install
# !pip install sageattention==2.2.0 --no-build-isolation

### 3e. FFmpeg

Required for muxing audio into the generated MP4. Most RunPod images already have it; this is a no-op if so.

In [8]:
# Try the system package first; fall back to a conda install if conda is available.
!ffmpeg -version >/dev/null 2>&1 && echo "ffmpeg already installed" || (apt-get update -y && apt-get install -y ffmpeg) || conda install -y -c conda-forge ffmpeg==7
!ffmpeg -version | head -n 1

ffmpeg already installed
ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers


## 4. Download the model weights

Two components are needed:

| Component | Repo | Local dir |
| :-- | :-- | :-- |
| SoulX-FlashHead 1.3B | `Soul-AILab/SoulX-FlashHead-1_3B` | `./models/SoulX-FlashHead-1_3B` |
| wav2vec2-base-960h | `facebook/wav2vec2-base-960h` | `./models/wav2vec2-base-960h` |

> **In mainland China**, uncomment the `HF_ENDPOINT` line to use the hf-mirror endpoint.

In [9]:
# import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"   # uncomment if in mainland China

!pip install -q "huggingface_hub[cli]"

!huggingface-cli download Soul-AILab/SoulX-FlashHead-1_3B --local-dir ./models/SoulX-FlashHead-1_3B
!huggingface-cli download facebook/wav2vec2-base-960h --local-dir ./models/wav2vec2-base-960h

print("\nModels directory:")
!ls -R ./models | head -n 40


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help


Hint: `hf` is already installed! Use it directly.

Hint: Examples:
  hf auth login
  hf download unsloth/gemma-4-31B-it-GGUF
  hf upload my-cool-model . .
  hf models ls --search "gemma"
  hf repos ls --format json
  hf jobs run python:3.12 python -c 'print("Hello!")'
  hf --help


Models directory:
ls: cannot access './models': No such file or directory


In [10]:
# import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"   # uncomment if in mainland China

!pip install -q -U huggingface_hub

!hf download Soul-AILab/SoulX-FlashHead-1_3B --local-dir ./models/SoulX-FlashHead-1_3B
!hf download facebook/wav2vec2-base-960h --local-dir ./models/wav2vec2-base-960h

print("\nModels directory:")
!ls -R ./models | head -n 40

Fetching 18 files: 100%|███████████████████████| 18/18 [02:02<00:00,  6.78s/it]
Download complete: 100%|██████████████████| 14.3G/14.3G [02:02<00:00, 3.00GB/s]✓ Downloaded
  path: /workspace/SoulX-FlashHead/models/SoulX-FlashHead-1_3B
Download complete: 100%|███████████████████| 14.3G/14.3G [02:02<00:00, 117MB/s]
Fetching 11 files: 100%|███████████████████████| 11/11 [00:08<00:00,  1.31it/s]
Download complete: 100%|███████████████████| 1.13G/1.13G [00:08<00:00, 261MB/s]✓ Downloaded
  path: /workspace/SoulX-FlashHead/models/wav2vec2-base-960h
Download complete: 100%|███████████████████| 1.13G/1.13G [00:08<00:00, 135MB/s]

Models directory:
./models:
SoulX-FlashHead-1_3B
wav2vec2-base-960h

./models/SoulX-FlashHead-1_3B:
Model_Lite
Model_Pro
README.md
VAE_LTX
VAE_Wan
assets
config.json
model_index.json

./models/SoulX-FlashHead-1_3B/Model_Lite:
config.json
diffusion_pytorch_model.safetensors

./models/SoulX-FlashHead-1_3B/Model_Pro:
config.json
diffusion_pytorch_model.safetensors

./mode

## 5. Inference

The cells below recreate the three inference scripts you provided and run them. They use the example inputs
shipped with the repo (`examples/girl.png` + `examples/podcast_sichuan_16k.wav`). To use your own image/audio,
edit the `--cond_image` / `--audio_path` arguments (a 16 kHz mono WAV works best).

`generate_video.py` flags used here:
- `--model_type` — `lite` or `pro`
- `--audio_encode_mode stream` — chunked streaming-style encoding
- `--cond_image` / `--audio_path` — the driving image and audio

We write the scripts to disk with `%%writefile` so the notebook is self-contained and matches the exact
scripts you uploaded.

### 5a. Lite model — single GPU

In [11]:
%%writefile inference_script_single_gpu_lite.sh
CUDA_VISIBLE_DEVICES=0

CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES python generate_video.py \
    --ckpt_dir models/SoulX-FlashHead-1_3B \
    --wav2vec_dir models/wav2vec2-base-960h \
    --model_type lite \
    --cond_image examples/girl.png \
    --audio_path examples/podcast_sichuan_16k.wav \
    --audio_encode_mode stream

Overwriting inference_script_single_gpu_lite.sh


In [ ]:
!sed -e 's/mediapipe==0.10.9/mediapipe>=0.10.13/' \
     -e '/nvidia-nccl-cu12/d' \
     requirements.txt > requirements_py312.txt
!pip install -r requirements_py312.txt --ignore-installed blinker
!python -c "import imageio, librosa, cv2, mediapipe; print('core deps OK')"

  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached opencv_python_headless-4.13.0.92-cp37-abi3-manylinux_2_28_x86_64.whl.metadata (19 kB)
  Using cached diffusers-0.38.0-py3-none-any.whl.metadata (20 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached tokenizers-0.23.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (9.8 kB)
  Using cached accelerate-1.13.0-py3-none-any.whl.metadata (19 kB)
  Using cached tqdm-4.68.1-py3-none-any.whl.metadata (57 kB)
  Using cached imageio-2.37.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached easydict-1.13-py3-none-any.whl.metadata (4.2 kB)
  Using cached ftfy-6.3.1-py3-none-any.whl.metadata (7.3 kB)
  Using cached imageio_ffmpeg-0.6.0-py3-none-manylinux2014_x86_64.whl.metadata (1.5 kB)
  Using cached scikit_image-0.26.0-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.wh

In [28]:
!pip install "transformers==4.57.3" "huggingface-hub>=0.34.0,<1.0" --no-deps
!python -c "import transformers, huggingface_hub; print('transformers', transformers.__version__); print('hf_hub', huggingface_hub.__version__)"

  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
Using cached transformers-4.57.3-py3-none-any.whl (12.0 MB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.2
    Uninstalling transformers-5.10.2:━━━━━━━━━━━ 0/2 [transformers]
      Successfully uninstalled transformers-5.10.20m 0/2 [transformers]
  Attempting uninstall: huggingface-hub━━━━━━━━━ 0/2 [transformers]
    Found existing installation: huggingface_hub 1.18.0/2 [transformers]
    Uninstalling huggingface_hub-1.18.0:━━━━ 0/2 [transformers]
      Successfully uninstalled huggingface_hub-1.18.0m0/2 [transformers]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [huggingface-hub] [huggingface-hub]
transformers 4.57.3
hf_hub 0.36.2


In [ ]:
!bash inference_script_single_gpu_lite.sh

### 5b. Pro model — single GPU

Higher quality, slower (~10.8 FPS on a 4090). Same single-GPU setup as Lite.

In [31]:
%%writefile inference_script_single_gpu_pro.sh
CUDA_VISIBLE_DEVICES=0

CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES python generate_video.py \
    --ckpt_dir models/SoulX-FlashHead-1_3B \
    --wav2vec_dir models/wav2vec2-base-960h \
    --model_type pro \
    --cond_image examples/girl.png \
    --audio_path examples/podcast_sichuan_16k.wav \
    --audio_encode_mode stream

Overwriting inference_script_single_gpu_pro.sh


In [ ]:
!bash inference_script_single_gpu_pro.sh

### 5c. Pro model — multi-GPU

⚠️ **Requires a pod with 2+ GPUs.** Uses `torchrun` across `GPU_NUM` devices. Real-time Pro speed is only
expected on two RTX 5090 with SageAttention installed (step 3d). Adjust `CUDA_VISIBLE_DEVICES` / `GPU_NUM`
to match your pod.

In [ ]:
%%writefile inference_script_multi_gpu_pro.sh
CUDA_VISIBLE_DEVICES=0,1
GPU_NUM=2
export NCCL_MIN_NCHANNELS=4

CUDA_VISIBLE_DEVICES=$CUDA_VISIBLE_DEVICES torchrun --nproc_per_node=$GPU_NUM generate_video.py \
    --ckpt_dir models/SoulX-FlashHead-1_3B \
    --wav2vec_dir models/wav2vec2-base-960h \
    --model_type pro \
    --cond_image examples/girl.png \
    --audio_path examples/podcast_sichuan_16k.wav \
    --audio_encode_mode stream

In [ ]:
# Run only if you have 2+ GPUs
import torch
if torch.cuda.device_count() >= 2:
    get_ipython().system('bash inference_script_multi_gpu_pro.sh')
else:
    print(f"Detected {torch.cuda.device_count()} GPU(s). Multi-GPU Pro needs 2+. Skipping.")

## 6. Preview the generated video

`generate_video.py` writes an MP4 into the repo. This cell finds the most recently created `.mp4` and
embeds it inline. If nothing shows up, check the cell output above for the exact save path.

In [ ]:
import glob, os
from IPython.display import Video, display

# Find the newest mp4 created under the repo (skip the bundled example clips)
mp4s = [p for p in glob.glob("**/*.mp4", recursive=True)]
mp4s = sorted(mp4s, key=lambda p: os.path.getmtime(p), reverse=True)

if mp4s:
    latest = mp4s[0]
    print("Most recent video:", os.path.abspath(latest))
    display(Video(latest, embed=True, width=512))
else:
    print("No .mp4 found yet. Run an inference cell above first.")

## 7. (Optional) Gradio demos

The repo ships two Gradio apps:

- `gradio_app.py` — standard demo (supports single- and multi-GPU via the UI)
- `gradio_app_streaming.py` — streaming demo, **single GPU only** (generate-and-play in real time)

On RunPod you can reach a Gradio app in two ways:

1. **Public share link** — easiest. Run with `share=True` (the cell below sets it via a tiny wrapper).
2. **RunPod proxy** — expose **HTTP port 7860** in your pod config, bind to `0.0.0.0:7860`, then open
   the pod's "Connect → HTTP Service [Port 7860]" link.

Because `app.launch()` blocks, run these in a **separate terminal** (RunPod web terminal) for the smoothest
experience, e.g. `cd /workspace/SoulX-FlashHead && python gradio_app.py`. The cell below shows the env-var
approach so the proxy/port binding is handled for you.

In [ ]:
# Configure host/port so the RunPod HTTP proxy (port 7860) can reach Gradio.
# Then launch in a terminal:  python gradio_app.py   (standard)  or  python gradio_app_streaming.py (streaming)
import os
os.environ["GRADIO_SERVER_NAME"] = "0.0.0.0"
os.environ["GRADIO_SERVER_PORT"] = "7860"
print("Gradio will bind to 0.0.0.0:7860 — expose HTTP port 7860 in your RunPod pod settings.")
print("To get a public link instead, run inside a terminal:  python gradio_app.py  with share=True in app.launch().")

# To launch the standard demo directly from the notebook (blocks the kernel — Stop to free it):
# !python gradio_app.py

# To launch the streaming demo (single GPU only):
# !python gradio_app_streaming.py

## 🛠️ Troubleshooting

- **FlashAttention build is slow / OOMs** → use the prebuilt wheel (step 3c) or lower `MAX_JOBS`.
- **CUDA / ABI mismatch errors** → make sure the torch from step 3a is active before installing `flash_attn`
  and `xformers`; reinstall in order if you changed torch.
- **`CUDA out of memory` on Pro** → use the Lite model, or a larger-VRAM / multi-GPU pod.
- **HF download stalls** → set `HF_ENDPOINT=https://hf-mirror.com` (step 4) or
  `export HF_HUB_ENABLE_HF_TRANSFER=1` for faster transfers.
- **No video produced** → re-check the inference cell output for the exact save path and any stack trace.
- **Gradio link unreachable** → confirm port 7860 is exposed in the pod, or use a `share=True` public link.

A network error mentioning a blocked domain means the relevant host isn't allowed in your pod's egress
settings — update the pod's network/firewall config accordingly.